# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata.id}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each RecordSet, Field, and Column is referenced by its `@id`. Let's enumerate the available record sets and their data structure.

In [ ]:
# List RecordSets available in the dataset
record_sets = dataset.record_sets
print("Available RecordSets:")
for rs in record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")

# For each RecordSet, list its fields and columns (@id)
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    print(" Fields:")
    for field in rs.fields:
        print(f"   - Field name: {field.name} | @id: {field.id} | dataType: {field.data_type}")
    print(" Columns:")
    for col in rs.columns:
        print(f"   - Column name: {col.name} | @id: {col.id} | field: {col.field.id if col.field else None}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will load each RecordSet's data into a pandas DataFrame for further exploration.

In [ ]:
# Prepare a list of RecordSet IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load records for each RecordSet
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display available columns for the first RecordSet
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's choose a RecordSet and fields for numeric analysis. For demonstration, we use the first RecordSet and pick numeric fields (e.g., GAD-7 score or PHQ-9 score) as observed in the metadata.

We perform:
  - Filtering (e.g., where GAD-7 > threshold)
  - Normalization
  - Grouping (e.g., by gender/education/village)

**Note:** Replace these with actual RecordSet and Field `@id`s as reported above.

In [ ]:
# Specify IDs based on the overview above
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Find a numeric field (assume 'gad7_score' or similar is present; ensure @id usage)
# We'll list columns to pick an appropriate one (replace as needed for real dataset):
print("Columns available:", df.columns.tolist())

# For example, let's suppose column '@id' is 'http://senscience.ai/gad7_score' (this must match schema)
numeric_field_id = 'http://senscience.ai/gad7_score'
if numeric_field_id in df.columns:
    numeric_field = numeric_field_id
else:
    # Fallback demo: pick first numeric column in schema
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        else:
            numeric_field = df.columns[0]

# Define some threshold for filtering
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field (e.g., gender or village -- must use @id)
# Let's suppose @id for gender is 'http://senscience.ai/gender', for village 'http://senscience.ai/village'
group_field_id = 'http://senscience.ai/gender'
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of GAD-7 scores, and compare groups by gender (if available).

In [ ]:
# Plot numeric field distribution
plt.figure(figsize=(8,4))
df[numeric_field].hist(bins=20)
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If gender field is available, plot boxplot
if group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df.boxplot(column=numeric_field, by=group_field_id)
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field)
    plt.title(f'{numeric_field} by {group_field_id}')
    plt.suptitle('')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrates loading, overview, and processing of the FAIR² Kilifi County Mental Health Survey Dataset via Croissant and `mlcroissant`.

- Loaded dataset and explored its metadata and structure via `@id`.
- Extracted tables using record set and field IDs.
- Applied filtering, normalization, and grouping to numeric survey scores.
- Visualized score distributions and compared demographic groupings.

This workflow enables standardized, reproducible AI-ready data exploration using Croissant schemas.